#Read Silver sellers

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "adb_ecommerce_de_7405607562556202"

sellers_df = spark.table(
    f"{catalog}.ecommerce_silver.sellers"
)

#Build seller dimensions

In [0]:
window_spec = Window.orderBy(sellers_df["seller_id"])

gold_dim_seller = (
    sellers_df
    .dropDuplicates(["seller_id"])
    .withColumn(
        "seller_key",
        F.row_number().over(window_spec)
    )
    .select(
        "seller_key",
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )
)

#Write to ADLS

In [0]:
gold_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/gold/dim_seller"

gold_dim_seller.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

#Register in Unity Catalog

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_gold.dim_seller
USING DELTA
LOCATION '{gold_path}'
""")

In [0]:
spark.table(
    f"{catalog}.ecommerce_gold.dim_seller"
).show(5, truncate=False)